In [5]:
import io
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from scipy.spatial.distance import euclidean

# Carregamento dos dados brutos fornecidos
raw_data = """[COLECAR OS DADOS AQUI]"""

# Leitura do DataFrame sem cabeçalho
df = pd.read_csv(io.StringIO(raw_data.strip()), header=None)

# 1. Descartar o atributo ID (primeira coluna)
df = df.drop(columns=[0])

# 2. Descartar instâncias (linhas) com dados ausentes ('?')
df = df.replace('?', np.nan).dropna()

# Converter todas as colunas para valores numéricos
for col in df.columns:
    df[col] = pd.to_numeric(df[col])

# Separação entre características (X) e rótulo alvo (y)
# O dataset original usa 2 para benigno e 4 para maligno
X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values
y_labels = np.array(['benign' if val == 2 else 'malignant' for val in y])

# Divisão em conjuntos de Treino e Teste (ex: 80% treino, 20% teste, estratificado)
X_train, X_test, y_train, y_test = train_test_split(
    X, y_labels, test_size=0.2, random_state=42, stratify=y_labels
)

# 3. Treinamento: Calcular o vetor médio (centróide) de cada classe
classes = np.unique(y_train)
centroids = {}
for cls in classes:
    centroids[cls] = np.mean(X_train[y_train == cls], axis=0)

# 4. Predição: Classificar instâncias de teste com base na Distância Euclidiana ao centróide mais próximo
predictions = []
for instance in X_test:
    min_dist = float('inf')
    pred_cls = None
    for cls, cent in centroids.items():
        dist = euclidean(instance, cent)
        if dist < min_dist:
            min_dist = dist
            pred_cls = cls
    predictions.append(pred_cls)

predictions = np.array(predictions)

# 5. Avaliação de Desempenho
correct_total = np.sum(predictions == y_test)
overall_acc = (correct_total / len(y_test)) * 100
print(f"Acurácia Geral: {overall_acc:.2f}%")

print("\nAcurácia por Classe:")
for cls in classes:
    mask = (y_test == cls)
    acc_cls = (np.sum(predictions[mask] == y_test[mask]) / np.sum(mask)) * 100
    print(f" - {cls.capitalize()}: {acc_cls:.2f}%")



Acurácia Geral: 96.35%

Acurácia por Classe:
 - Benign: 95.51%
 - Malignant: 97.92%
